# 🧱 Lab: Triage CSV Rows with an AI Pipeline

**Module 1: Ship a Working AI Feature Today** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Run** a complete AI pipeline over real CSV rows, not a single pasted message
2. **Collect** structured results into a list of dicts and export a v0 checkpoint
3. **Record** stage status, errors, a baseline score and open questions in that v0 checkpoint

## Concepts Covered

| Concept | Gloss | From |
|---------|-------|------|
| AI Pipeline Stages | Ingest, prepare, call, validate, log, evaluate; model is one box | mini-run-first-triage |

## Prerequisites

- **mini-run-first-triage**: Run the mock pipeline on one message and map the six boxes

## Setup

We call the LLM with the OpenAI SDK, so put your API key in a `.env` file.

In [6]:
import os
import csv
import json
from io import StringIO
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv()

client = OpenAI()

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## The Task: Customer Support Ticket Triage

You're building an AI feature that reads customer support tickets from a CSV and classifies each one into a category: **Billing**, **Technical**, **Account**, or **General**. This is exactly the kind of feature an AI engineer ships in a real product.

We'll build it from six small functions, then deliberately break it at different steps to practice failure localization. Each function sits in one of the six boxes:

<div style='display:flex;flex-wrap:wrap;align-items:center;gap:8px;margin:12px 0 16px 0;'>
<span style='border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>1. Ingest</strong><br>ingest()</span>
<span>→</span>
<span style='border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>2. Prepare</strong><br>prepare() + build_prompt()</span>
<span>→</span>
<span style='border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>3. Call</strong><br>call_model()</span>
<span>→</span>
<span style='border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>4. Validate</strong><br>parse_output()</span>
<span>→</span>
<span style='border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>5. Log</strong><br>deliver()</span>
<span>→</span>
<span style='border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>6. Evaluate</strong><br>v0 spot-check</span>
</div>

The headings below number the six functions in order, so their stage numbers don't match the box numbers: "Stage 3 — Prompt" is the second half of **Prepare**, "Stage 4 — Call Model" is **Call**, "Stage 5 — Parse Output" is **Validate**, and "Stage 6 — Deliver" is **Log**.


## Stage 1 — Ingest: Create and Load the CSV

First, we simulate a small CSV of support tickets. In production this would be a database query or file upload.

In [7]:
# Create a small CSV of support tickets
CSV_DATA = """ticket_id,customer,message
101,Alice,"I was charged twice for my subscription last month. Please refund the duplicate."
102,Bob,"The app crashes every time I try to upload a file larger than 10MB."
103,Carol,"How do I change the email address on my account?"
104,Dan,"I love your product! Just wanted to say thanks to the team."
105,Eve,"My invoice shows a charge for a plan I never signed up for."
106,Frank,"Getting a 502 error on the API endpoint since this morning."
"""

os.makedirs("data", exist_ok=True)
csv_path = os.path.join("data", "tickets.csv")

with open(csv_path, "w", newline="") as f:
    f.write(CSV_DATA.strip())

print(f"✅ Wrote {csv_path}")

✅ Wrote data\tickets.csv


In [10]:
# Stage 1: Ingest — read the CSV into a list of dicts
def ingest(path):
    """Stage 1: Load tickets from a CSV file."""
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        tickets = list(reader)
    print(f"Stage 1 (Ingest): Loaded {len(tickets)} tickets from {path}")
    return tickets

tickets = ingest(csv_path)
tickets[0]

Stage 1 (Ingest): Loaded 6 tickets from data\tickets.csv


{'ticket_id': '101',
 'customer': 'Alice',
 'message': 'I was charged twice for my subscription last month. Please refund the duplicate.'}

## Stage 2 — Prepare: Clean and Validate Rows

Before sending anything to a model, we validate that each row has the fields we need and the message isn't empty.

In [12]:
# Stage 2: Prepare — validate and clean each ticket
def prepare(tickets):
    """Stage 2: Validate tickets and strip whitespace."""
    clean = []
    for t in tickets:
        msg = t.get("message", "").strip()
        if not msg:
            print(f"  ⚠️  Skipping ticket {t.get('ticket_id', '?')}: empty message")
            continue
        clean.append({"ticket_id": t["ticket_id"], "customer": t["customer"], "message": msg})
    print(f"Stage 2 (Prepare): {len(clean)} tickets ready (out of {len(tickets)})")
    return clean

clean_tickets = prepare(tickets)
print(f"First clean ticket: {clean_tickets[0]}")

Stage 2 (Prepare): 6 tickets ready (out of 6)
First clean ticket: {'ticket_id': '101', 'customer': 'Alice', 'message': 'I was charged twice for my subscription last month. Please refund the duplicate.'}


## Stage 3 — Prompt: Build the Classification Prompt

We construct a system prompt that tells the model exactly what to do, and a user prompt for each ticket.

In [13]:
SYSTEM_PROMPT = """You are a support ticket classifier. For each customer message, respond with ONLY a JSON object:
{"category": "<one of: Billing, Technical, Account, General>", "confidence": "<high, medium, or low>"}

Rules:
- Billing: charges, refunds, invoices, payments
- Technical: bugs, errors, crashes, API issues
- Account: email, password, profile, settings changes
- General: everything else (feedback, questions, greetings)
"""

def build_prompt(ticket):
    """Stage 3: Assemble messages for the LLM."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT.strip()},
        {"role": "user", "content": f"Classify this ticket:\n\n{ticket['message']}"}
    ]

# Preview the user prompt built for the first ticket
sample_messages = build_prompt(clean_tickets[0])
print(f"Stage 3 (Prompt): ticket {clean_tickets[0]['ticket_id']} →", sample_messages[1]["content"])

Stage 3 (Prompt): ticket 101 → Classify this ticket:

I was charged twice for my subscription last month. Please refund the duplicate.


## Stage 4 — Call Model: Send to the LLM

We call the OpenAI API for each ticket. In production you'd batch these or use async, but for learning we keep it simple.

In [16]:
def call_model(messages):
    """Stage 4: Send prompt to LLM and return raw response text."""
    response = client.chat.completions.create(
        model="gpt-5.6-luna",
        messages=messages,
        reasoning_effort="none",
    )
    return response.choices[0].message.content

# Test with first ticket
raw_output = call_model(sample_messages)
print("Stage 4 (Call Model): Raw response →", raw_output)

Stage 4 (Call Model): Raw response → {"category":"Billing","confidence":"high"}


## Stage 5 — Parse Output: Extract Structured Data

The model returns a JSON string. We parse it into a Python dict. This is where many real pipelines break — the model might return extra text or malformed JSON.

In [17]:
def parse_output(raw_text):
    """Stage 5: Parse the model's JSON response."""
    result = json.loads(raw_text)
    # Validate expected fields exist
    assert "category" in result, "Missing 'category' field"
    assert "confidence" in result, "Missing 'confidence' field"
    return result

parsed = parse_output(raw_output)
print("Stage 5 (Parse Output):", parsed)

Stage 5 (Parse Output): {'category': 'Billing', 'confidence': 'high'}


## Stage 6 — Deliver: Collect Results

In a real app you'd write to a database, send a webhook, or update a UI. Here we collect results into a list for display.

In [18]:
def deliver(ticket, classification):
    """Stage 6: Format the final result for delivery."""
    return {
        "ticket_id": ticket["ticket_id"],
        "customer": ticket["customer"],
        "category": classification["category"],
        "confidence": classification["confidence"],
    }

# Test with first ticket
result = deliver(clean_tickets[0], parsed)
print("Stage 6 (Deliver):", result)

Stage 6 (Deliver): {'ticket_id': '101', 'customer': 'Alice', 'category': 'Billing', 'confidence': 'high'}


## Run the Full Pipeline

Now we chain all six stages together and process every ticket. This is the end-to-end run.

In [19]:
def run_pipeline(csv_path):
    """Run the full six-stage triage pipeline."""
    # Stage 1: Ingest
    tickets = ingest(csv_path)
    
    # Stage 2: Prepare
    clean = prepare(tickets)
    
    results = []
    for ticket in clean:
        # Stage 3: Prompt
        messages = build_prompt(ticket)
        
        # Stage 4: Call Model
        raw = call_model(messages)
        
        # Stage 5: Parse Output
        classification = parse_output(raw)
        
        # Stage 6: Deliver
        results.append(deliver(ticket, classification))
    
    return results

results = run_pipeline(csv_path)

# Each result is a plain dict: count them all and peek at the first two
print(f"✅ Triaged {len(results)} tickets")
for r in results[:2]:
    print(r)

Stage 1 (Ingest): Loaded 6 tickets from data\tickets.csv
Stage 2 (Prepare): 6 tickets ready (out of 6)
✅ Triaged 6 tickets
{'ticket_id': '101', 'customer': 'Alice', 'category': 'Billing', 'confidence': 'high'}
{'ticket_id': '102', 'customer': 'Bob', 'category': 'Technical', 'confidence': 'high'}


In [20]:
# Check a specific ticket's result
for r in results:
    if r['customer'] == 'Frank':
        print(f"Frank's ticket: {r['category']} ({r['confidence']})")

Frank's ticket: Technical (high)


## Break It: Failure Localization Practice

The real skill of working with AI pipelines is knowing **which stage broke**. We'll deliberately introduce failures at different stages, catch them, and name the stage.

We'll use a wrapper function that catches errors and maps them to pipeline stages.

In [21]:
def diagnose_stage(stage_num, stage_name, error):
    """Format a stage-failure diagnosis."""
    return (
        f"💥 **Pipeline broke at Stage {stage_num} — {stage_name}**\n\n"
        f"Error type: `{type(error).__name__}`\n\n"
        f"Message: `{error}`\n\n"
        f"*Fix:* Look at your **{stage_name}** code, not the model or the prompt."
    )

### Break Stage 1 — Ingest

What happens when the file doesn't exist?

In [22]:
# Deliberately pass a wrong file path
try:
    ingest("data/nonexistent.csv")
except FileNotFoundError as e:
    md(diagnose_stage(1, "Ingest", e))

💥 **Pipeline broke at Stage 1 — Ingest**

Error type: `FileNotFoundError`

Message: `[Errno 2] No such file or directory: 'data/nonexistent.csv'`

*Fix:* Look at your **Ingest** code, not the model or the prompt.

Stage 2 (Prepare) needs no break: `prepare()` already skips rows whose message is missing or empty. So we move on to Stage 3.

### Break Stage 3 — Prompt

What happens when you reference a field that doesn't exist in the ticket dict?

In [23]:
# A build_prompt that references a field that doesn't exist
def build_prompt_broken(ticket):
    """A broken version that references a non-existent field."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT.strip()},
        {"role": "user", "content": f"Classify: {ticket['subject']}"} # 'subject' doesn't exist!
    ]

try:
    build_prompt_broken(clean_tickets[0])
except KeyError as e:
    md(diagnose_stage(3, "Prompt", e))

💥 **Pipeline broke at Stage 3 — Prompt**

Error type: `KeyError`

Message: `'subject'`

*Fix:* Look at your **Prompt** code, not the model or the prompt.

### Break Stage 5 — Parse Output

What happens when the model returns something that isn't valid JSON?

In [24]:
# Simulate a model returning malformed output
bad_response = "I'd classify this as Billing with high confidence."

try:
    parse_output(bad_response)
except json.JSONDecodeError as e:
    md(diagnose_stage(5, "Parse Output", e))

💥 **Pipeline broke at Stage 5 — Parse Output**

Error type: `JSONDecodeError`

Message: `Expecting value: line 1 column 1 (char 0)`

*Fix:* Look at your **Parse Output** code, not the model or the prompt.

In [25]:
# What about extra text around valid JSON?
try:
    parse_output('Sure! Here is the result: {"category": "Billing", "confidence": "high"}')
except json.JSONDecodeError as e:
    md(diagnose_stage(5, "Parse Output", e))

💥 **Pipeline broke at Stage 5 — Parse Output**

Error type: `JSONDecodeError`

Message: `Expecting value: line 1 column 1 (char 0)`

*Fix:* Look at your **Parse Output** code, not the model or the prompt.

## Pipeline with Stage Labels

Let's build a hardened version of the pipeline that catches errors at each stage and tells you exactly where it broke. This is the pattern you'll use in real AI engineering work.

In [26]:
def run_pipeline_safe(csv_path):
    """Run the pipeline with per-stage error handling."""
    # Stage 1
    try:
        tickets = ingest(csv_path)
    except Exception as e:
        md(diagnose_stage(1, "Ingest", e))
        return []

    # Stage 2
    try:
        clean = prepare(tickets)
    except Exception as e:
        md(diagnose_stage(2, "Prepare", e))
        return []

    results = []
    for ticket in clean:
        try:
            messages = build_prompt(ticket)                  # Stage 3
            raw = call_model(messages)                       # Stage 4
            classification = parse_output(raw)               # Stage 5
            results.append(deliver(ticket, classification))  # Stage 6
        except Exception as e:
            # One broad catch per ticket: the error type points you to the stage
            print(f"⚠️ Ticket {ticket['ticket_id']} failed: {type(e).__name__}: {e}")
            print("   Hint: KeyError → Prompt, JSONDecodeError/AssertionError → Parse Output, API error → Call Model")

    return results

safe_results = run_pipeline_safe(csv_path)
print(f"\n✅ Successfully triaged {len(safe_results)} tickets.")

Stage 1 (Ingest): Loaded 6 tickets from data\tickets.csv
Stage 2 (Prepare): 6 tickets ready (out of 6)

✅ Successfully triaged 6 tickets.


---
## v0 recap

A short checkpoint of where the pipeline stands. This is not a separate lab.

## What Is a v0 Checkpoint?

A **v0 checkpoint** is the first snapshot of your pipeline, the one you share to say *"here's where we are after the first pass."* It has four parts: **stage status** (what works), **errors** mapped to stages (what breaks), a **baseline** spot-check score (how good it is), and **open questions** for the next version.

## Step 1: Record Pipeline Stage Status

Recall the six boxes from `mini-run-first-triage`. For your v0 checkpoint, you walk through each step of this pipeline and record whether it ran successfully, failed, or wasn't attempted yet.

Let's record the status of the ticket-triage pipeline you just built, after its first pass.

In [27]:
# Record the status of each pipeline stage after your first end-to-end run
stage_status = [
    {"stage": 1, "name": "Ingest",       "status": "✅ Pass", "notes": "CSV loads correctly, 6 tickets read"},
    {"stage": 2, "name": "Prepare",      "status": "✅ Pass", "notes": "Empty messages skipped, 6/6 passed"},
    {"stage": 3, "name": "Prompt",       "status": "✅ Pass", "notes": "System + user messages built for each ticket"},
    {"stage": 4, "name": "Call Model",   "status": "✅ Pass", "notes": "gpt-5.6-luna, reasoning_effort=none"},
    {"stage": 5, "name": "Parse Output", "status": "⚠️ Fragile", "notes": "Works now, but no fallback if model returns non-JSON"},
    {"stage": 6, "name": "Deliver",      "status": "✅ Pass", "notes": "Results collected into a list of dicts"},
]

header = "| Stage | Name | Status | Notes |\n|-------|------|--------|-------|\n"
rows = ""
for s in stage_status:
    rows += f"| {s['stage']} | {s['name']} | {s['status']} | {s['notes']} |\n"

md("### Pipeline Stage Status\n\n" + header + rows)

### Pipeline Stage Status

| Stage | Name | Status | Notes |
|-------|------|--------|-------|
| 1 | Ingest | ✅ Pass | CSV loads correctly, 6 tickets read |
| 2 | Prepare | ✅ Pass | Empty messages skipped, 6/6 passed |
| 3 | Prompt | ✅ Pass | System + user messages built for each ticket |
| 4 | Call Model | ✅ Pass | gpt-5.6-luna, reasoning_effort=none |
| 5 | Parse Output | ⚠️ Fragile | Works now, but no fallback if model returns non-JSON |
| 6 | Deliver | ✅ Pass | Results collected into a list of dicts |


## Step 2: Record Errors Encountered (with Stage Labels)

Your v0 checkpoint should list every error you hit during development, mapped to the step it came from. This creates a debugging log for your future self.

In [28]:
# Log of errors encountered during development, mapped to pipeline stages
errors_encountered = [
    {
        "stage": 1,
        "stage_name": "Ingest",
        "error_type": "FileNotFoundError",
        "description": "Typo in CSV path ('data/tickers.csv' instead of 'data/tickets.csv')",
        "resolved": True,
    },
    {
        "stage": 3,
        "stage_name": "Prompt",
        "error_type": "KeyError",
        "description": "Referenced ticket['subject'] instead of ticket['message']",
        "resolved": True,
    },
    {
        "stage": 5,
        "stage_name": "Parse Output",
        "error_type": "JSONDecodeError",
        "description": "Model occasionally returns 'Sure! Here is...' prefix before JSON",
        "resolved": False,
    },
]

resolved_count = sum(1 for e in errors_encountered if e["resolved"])
total_count = len(errors_encountered)

# The full errors table is rendered in the assembled checkpoint below
md(f"**Logged {total_count} errors ({resolved_count} resolved, {total_count - resolved_count} still open)**")

**Logged 3 errors (2 resolved, 1 still open)**

## Step 3: Record Baseline Accuracy

"Looks good" isn't enough. Your v0 checkpoint records an actual score: the model's labels checked against labels a person wrote for a few tickets.

In [29]:
# Baseline spot-check: the model's labels against labels a person wrote for five tickets
baseline_results = [
    {"ticket_id": "101", "expected": "Billing",   "predicted": "Billing",   "match": True},
    {"ticket_id": "102", "expected": "Technical", "predicted": "Technical", "match": True},
    {"ticket_id": "103", "expected": "Account",   "predicted": "General",   "match": False},
    {"ticket_id": "104", "expected": "General",   "predicted": "General",   "match": True},
    {"ticket_id": "105", "expected": "Billing",   "predicted": "Billing",   "match": True},
]

correct = sum(1 for r in baseline_results if r["match"])
total = len(baseline_results)
accuracy = correct / total

# Show the score plus the first miss, which is the example worth looking at
miss = [r for r in baseline_results if not r["match"]][0]
md(
    f"**Baseline accuracy: {correct}/{total} ({accuracy:.0%})**. "
    f"Example miss: ticket {miss['ticket_id']} expected {miss['expected']}, got {miss['predicted']}"
)

**Baseline accuracy: 4/5 (80%)**. Example miss: ticket 103 expected Account, got General

## Step 4: Write Open Questions

The most underrated part of a checkpoint is the **open questions list**. These are things you don't know yet — unknowns that would change your approach if you had answers. Writing them down prevents you from forgetting and gives your team a clear picture of what remains uncertain.

In [30]:
# Open questions: things you'd want to answer before v1
open_questions = [
    "The model confused 'Account' with 'General' on ticket 103. Would a few-shot example for Account changes fix this?",
    "Parse Output has no fallback. What happens if the model wraps JSON in markdown fences (```json ... ```)?",
    "Tickets are processed one at a time. Will this be too slow for 1,000 tickets?",
]

md("### Open Questions\n\n" + "\n".join(f"- {q}" for q in open_questions))

### Open Questions

- The model confused 'Account' with 'General' on ticket 103. Would a few-shot example for Account changes fix this?
- Parse Output has no fallback. What happens if the model wraps JSON in markdown fences (```json ... ```)?
- Tickets are processed one at a time. Will this be too slow for 1,000 tickets?

## Step 5: Assemble the v0 Checkpoint

Now we combine all four pieces into a single checkpoint artifact — one document you could save, print, or paste into a team thread.

In [31]:
from datetime import date

today = date.today().isoformat()

# Header + Section 1: stage status table
checkpoint_text = (
    f"# v0 Checkpoint: Ticket Triage Pipeline\n\n"
    f"**Date:** {today} | **Model:** gpt-5.6-luna (reasoning_effort=none)\n\n"
    "## 1. Pipeline Stage Status\n\n"
    "| Stage | Name | Status | Notes |\n|-------|------|--------|-------|\n"
)
for s in stage_status:
    checkpoint_text += f"| {s['stage']} | {s['name']} | {s['status']} | {s['notes']} |\n"

# Section 2: errors table, each row labeled with its stage
checkpoint_text += f"\n## 2. Errors Encountered ({resolved_count}/{total_count} resolved)\n\n"
checkpoint_text += "| Stage | Error Type | Description | Resolved? |\n|-------|------------|-------------|-----------|\n"
for e in errors_encountered:
    icon = "✅" if e["resolved"] else "❌"
    checkpoint_text += f"| {e['stage']} — {e['stage_name']} | `{e['error_type']}` | {e['description']} | {icon} |\n"

# Section 3: baseline as one line
checkpoint_text += f"\n## 3. Baseline Accuracy\n\n{correct}/{total} ({accuracy:.0%}) on a hand-labeled spot-check\n"

# Section 4: open questions as bullets
checkpoint_text += "\n## 4. Open Questions\n\n"
for q in open_questions:
    checkpoint_text += f"- {q}\n"

md(checkpoint_text)

# v0 Checkpoint: Ticket Triage Pipeline

**Date:** 2026-10-29 | **Model:** gpt-5.6-luna (reasoning_effort=none)

## 1. Pipeline Stage Status

| Stage | Name | Status | Notes |
|-------|------|--------|-------|
| 1 | Ingest | ✅ Pass | CSV loads correctly, 6 tickets read |
| 2 | Prepare | ✅ Pass | Empty messages skipped, 6/6 passed |
| 3 | Prompt | ✅ Pass | System + user messages built for each ticket |
| 4 | Call Model | ✅ Pass | gpt-5.6-luna, reasoning_effort=none |
| 5 | Parse Output | ⚠️ Fragile | Works now, but no fallback if model returns non-JSON |
| 6 | Deliver | ✅ Pass | Results collected into a list of dicts |

## 2. Errors Encountered (2/3 resolved)

| Stage | Error Type | Description | Resolved? |
|-------|------------|-------------|-----------|
| 1 — Ingest | `FileNotFoundError` | Typo in CSV path ('data/tickers.csv' instead of 'data/tickets.csv') | ✅ |
| 3 — Prompt | `KeyError` | Referenced ticket['subject'] instead of ticket['message'] | ✅ |
| 5 — Parse Output | `JSONDecodeError` | Model occasionally returns 'Sure! Here is...' prefix before JSON | ❌ |

## 3. Baseline Accuracy

4/5 (80%) on a hand-labeled spot-check

## 4. Open Questions

- The model confused 'Account' with 'General' on ticket 103. Would a few-shot example for Account changes fix this?
- Parse Output has no fallback. What happens if the model wraps JSON in markdown fences (```json ... ```)?
- Tickets are processed one at a time. Will this be too slow for 1,000 tickets?


In [32]:
print(checkpoint_text[:200])

# v0 Checkpoint: Ticket Triage Pipeline

**Date:** 2026-10-29 | **Model:** gpt-5.6-luna (reasoning_effort=none)

## 1. Pipeline Stage Status

| Stage | Name | Status | Notes |
|-------|------|--------


## Save the Checkpoint

A checkpoint is only useful if it's saved somewhere. Let's write it to a file.

In [33]:
import os

os.makedirs("data", exist_ok=True)
checkpoint_path = os.path.join("data", "checkpoint_v0.md")

# encoding="utf-8" is required on Windows: the default (cp1252) cannot
# write emoji such as ✅ / ❌ / ⚠️ that appear in checkpoint_text.
with open(checkpoint_path, "w", encoding="utf-8") as f:
    f.write(checkpoint_text)

print(f"✅ Checkpoint saved to {checkpoint_path}")

✅ Checkpoint saved to data\checkpoint_v0.md


## Why This Matters

Without a saved checkpoint, a week later you can't remember:
- Which stages you validated and which errors are still open
- What your accuracy actually was

The checkpoint is the difference between *"I think it was working"* and *"Here's exactly where we are."*

## 🎯 Summary

### What You Built

You built a complete AI-powered ticket triage pipeline that reads customer support tickets from a CSV, classifies each one via an LLM, and collects structured results — then you deliberately broke it at multiple stages and practiced diagnosing exactly where each failure occurred.

### Key Takeaways

1. **End-to-end pipeline** — even a simple AI feature like ticket classification touches all six boxes: ingest → prepare → call → validate → log → evaluate
2. **Stage isolation** — wrapping each stage in its own function makes failures easy to locate and fix independently
3. **Parse Output is fragile** — the model can return text that looks right but isn't valid JSON; this is the most common real-world pipeline breakage
4. **Failure localization** — when something breaks, asking "which of the six stages broke?" is faster than staring at the whole pipeline
5. **v0 checkpoint** — stage status, errors, a spot-check score and open questions, saved to one file, show exactly where the project stands